# Reportes de Desempeño
***Data Understanding y Data Preparation***

## 1. Configuración del entorno

In [1]:
import glob
import json
import os

import pandas as pd

from google.colab import drive
RUTA_DRIVE = "/content/drive/My Drive"

if not os.path.isdir(RUTA_DRIVE):
    drive.mount("/content/drive")

RUTA_PERF = f"{RUTA_DRIVE}/Datos/Desempeno"

Mounted at /content/drive


## 2. Localización y carga de reportes
Se localiza el JSON más reciente de cada notebook y se carga su contenido.

In [2]:
def ultimo(prefijo):
    archivos = sorted(glob.glob(f"{RUTA_PERF}/{prefijo}_*.json"))
    if not archivos:
        raise FileNotFoundError(f"No hay archivos '{prefijo}_*.json' en {RUTA_PERF}")
    return archivos[-1]            # el run_id es fecha_hora: el último en orden es el más reciente


def cargar(ruta):
    with open(ruta, encoding="utf-8") as fh:
        return json.load(fh)

ruta1 = ultimo("desempeno_nb1")
ruta2 = ultimo("desempeno_nb2")

rep1 = cargar(ruta1)
rep2 = cargar(ruta2)

print(f"DataUnderstanding: {os.path.basename(ruta1)}")
print(f"DataPreparation:  {os.path.basename(ruta2)}")

if rep2["run_id"] < rep1["run_id"]:
    print(
        "AVISO: el reporte de DataPreparation es más antiguo "
        "que el de DataUnderstanding; podrían no corresponder "
        "a la misma corrida."
    )

DataUnderstanding: desempeno_nb1_20261029_214202.json
DataPreparation:  desempeno_nb2_20261029_215115.json


## 3. Integración de etapas y cálculo de indicadores

In [3]:
etapas = (
    [{**e, "notebook": "DataUnderstanding"} for e in rep1["etapas"]] +
    [{**e, "notebook": "DataPreparation"} for e in rep2["etapas"]]
)

pipe = [e for e in etapas if e["categoria"] != "exploratorio"]
total = sum(e["segundos"] for e in pipe)
top = max(pipe, key=lambda e: e["segundos"])

pipeline = {
    "run_id_nb1": rep1["run_id"],
    "run_id_nb2": rep2["run_id"],
    "entorno": rep2["entorno"],
    "resumen": {
        "segundos_pipeline_total": round(total, 1),
        "segundos_pipeline_por_notebook": {
            "DataUnderstanding": rep1["resumen"]["segundos_pipeline"],
            "DataPreparation": rep2["resumen"]["segundos_pipeline"]
        },
        "segundos_exploratorio_total": round(
            sum(e["segundos"] for e in etapas if e["categoria"] == "exploratorio"), 1
        ),
        "etapa_mas_lenta": f"{top['notebook']}.{top['nombre']}",
        "pct_etapa_mas_lenta": round(top["segundos"] / total * 100, 1),
        "rss_pico_mb": max(e["rss_pico_mb"] for e in etapas)},
    "etapas": etapas
}

## 4. Exportación del reporte

In [4]:
salida = f"{RUTA_PERF}/desempeno_pipeline_{rep2['run_id']}.json"


with open(salida, "w", encoding="utf-8") as fh:
    json.dump(
        pipeline,
        fh,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

if os.path.isfile(salida) and os.path.getsize(salida) > 0:
    print(f"Reporte guardado correctamente: {salida}")
    print(f"Tamaño: {os.path.getsize(salida):,} bytes")
else:
    raise IOError(f"No se pudo verificar el archivo: {salida}")

Reporte guardado correctamente: /content/drive/My Drive/Datos/Desempeno/desempeno_pipeline_20261029_215115.json
Tamaño: 8,835 bytes


## 5. Resumen y detalle de performance

In [5]:
resumen = pipeline["resumen"]

print(f"Tiempo total del pipeline: "
      f"{resumen['segundos_pipeline_total']:.1f} s")

print(
    f"Etapa más lenta: {resumen['etapa_mas_lenta']} "
    f"({resumen['pct_etapa_mas_lenta']} %)"
)

print(
    f"Tiempo exploratorio total: "
    f"{resumen['segundos_exploratorio_total']:.1f} s"
)

print(f"Pico de RSS: {resumen['rss_pico_mb']} MB")

display(pd.DataFrame(etapas)[[
    "notebook",
    "nombre",
    "categoria",
    "segundos",
    "filas",
    "filas_por_s",
    "rss_pico_mb",
]])

Tiempo total del pipeline: 378.0 s
Etapa más lenta: DataUnderstanding.leer_xlsx_historico (67.0 %)
Tiempo exploratorio total: 4.8 s
Pico de RSS: 3106 MB


,notebook,nombre,categoria,segundos,filas,filas_por_s,rss_pico_mb
0,DataUnderstanding,leer_csv_2025_y_consolidar,io_lectura,13.729,7249,528,181
1,DataUnderstanding,leer_xlsx_historico,io_lectura,253.185,408879,1615,309
2,DataUnderstanding,anonimizar_historico,transformacion,4.902,408879,83413,402
3,DataUnderstanding,consolidar_historico,transformacion,0.332,278767,840763,501
4,DataUnderstanding,conversion_tipos,transformacion,5.534,278767,50371,725
5,DataUnderstanding,calidad_datos,validacion,7.512,278767,37110,1079
6,DataUnderstanding,tiempo_libre_cola,transformacion,0.979,278767,284812,1216
7,DataUnderstanding,reconstruccion_lactancia,transformacion,0.834,278767,334335,2049
8,DataUnderstanding,S0_base_limpia,transformacion,1.405,278167,198043,2457
9,DataUnderstanding,S3_episodios_cola,transformacion,0.234,278167,1188294,2355
